# One agent, three frameworks, one diff

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 18 §18.7 · type: walkthrough*

Build the **same** research agent — `search_docs` → a cited answer — three ways (raw Anthropic
SDK, LangGraph, Pydantic AI), then read them side by side to see exactly what each framework
gives you and what it hides.


## 🧠 Why this matters

A framework is a *loan of architecture*: you get a working structure now and repay in
flexibility later. Every framework sells the same five things — the loop, state management,
integrations, orchestration primitives, and the production rim (streaming, retries,
human-in-the-loop, tracing). The only way to evaluate that loan is to know what it hides, and
the fastest way to know is to build one agent with no framework and then watch two frameworks
absorb the same agent. After Ch 12–17 you've already written every internal these frameworks
package — so this notebook is about *reading* the trade-off in code, not being impressed by it.


## Objectives & prerequisites

**By the end you can:**

- Implement the same `search_docs` research agent in raw SDK, LangGraph, and Pydantic AI.
- Point at the exact line where each framework *hides* something from a debugger.
- Articulate, in code, what you delegate to a framework and what it costs to leave.

**Prereqs (concepts, not installs):**

- **Ch 12** — the raw tool loop you own end-to-end.
- **Ch 13** — the `search_docs` retriever (here mocked over a tiny doc set).
- **Ch 14** — checkpoints (what LangGraph's checkpointer automates).
- **Ch 15** — structured outputs (what Pydantic AI validates at the boundary).

This notebook runs **free and offline** in mock mode — no API key required.


## Setup

`MOCK=1` (the default) returns a canned, deterministic tool-use + final answer so the three
implementations produce the **same** cited answer and the diff is purely structural. Set
`COMPANION_MOCK=0` *and* export `ANTHROPIC_API_KEY` to hit the live API (≈ one short run per
framework; the point is *shape*, not output).


In [ ]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default): canned, offline, deterministic. MOCK=0: live API (needs a key).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed everything stochastic. (LLM sampling itself is not fully deterministic in MOCK=0.)
random.seed(18)

MODEL = "claude-opus-4-8"  # the book's default; verify the current id against the docs

if not MOCK and not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
        "Export the key or unset COMPANION_MOCK to run offline."
    )

print(f"MOCK={MOCK}  model={MODEL}")


## The shared contract (define once, reuse three times)

Every framework below answers the **same** task over the **same** tool. Pinning the contract
here is the whole experiment: change the framework, hold everything else fixed, read the diff.

- **Task:** *Answer from the document base. Cite source ids. Say so if the documents don't cover it.*
- **Tool:** `search_docs(query)` — the Ch 13 retriever, here a tiny keyword match over `data/docs.json`.


In [ ]:
DOCS = json.loads((Path("data") / "docs.json").read_text(encoding="utf-8"))["docs"]

SYSTEM = (
    "Answer from the document base. Cite source ids. "
    "Say so if the documents don't cover it."
)
QUESTION = "How long do customers have to request a refund, and how fast is the refund paid?"


def rag_search(query: str) -> str:
    """The Ch 13 retriever, shrunk to a deterministic keyword match.

    Returns matching passages, each prefixed with its source id so the model can cite it.
    """
    q = query.lower()
    words = [w for w in q.split() if len(w) > 3]
    hits = [
        d for d in DOCS
        if any(w in d["text"].lower() or w in d["title"].lower() for w in words)
    ]
    hits = hits or DOCS[:1]  # never return nothing; the model decides relevance
    return "\n".join(f"[{d['id']}] {d['text']}" for d in hits[:3])


print(rag_search("refund window and payment time"))


### The canned run (so MOCK is realistic, not empty)

In mock mode all three frameworks resolve to the same two-step transcript: the model calls
`search_docs` once, reads the passages, then returns a cited answer. We model that once and
reuse it everywhere — identical output, three shapes.


In [ ]:
# The deterministic answer every framework converges on in MOCK=1.
CANNED_QUERY = "refund request window and payout time"
CANNED_ANSWER = (
    "Customers have 30 days from purchase to request a refund. Once approved, "
    "the refund is paid to the original payment method within 5-7 business days. [doc-refunds-01]"
)
CANNED_SOURCE_IDS = ["doc-refunds-01"]
CANNED_CONFIDENCE = 0.9


def line_count(src: str) -> int:
    """Non-blank lines of a code string — our crude 'how much did you write' meter."""
    return sum(1 for ln in src.strip().splitlines() if ln.strip())


## 1 · Raw Anthropic SDK — you own the loop

Nothing hides. You declare the tool schema, run the `while` loop, append `tool_result` blocks,
and stop when `stop_reason != "tool_use"`. This is the most code and the most honest code —
every message and turn is visible to a debugger. It matches the §18.7 sketch.


In [ ]:
import anthropic  # noqa: F401  (imported to mirror the real shape; only used when MOCK=0)

TOOLS = [{
    "name": "search_docs",
    "description": "Search the company document base. Returns passages with source ids.",
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
}]


def research_raw(question: str) -> str:
    if MOCK:
        # The loop, made explicit: one tool turn, then the final text.
        _passages = rag_search(CANNED_QUERY)        # turn 1: model asked for a search
        return CANNED_ANSWER                        # turn 2: model answered from passages

    client = anthropic.Anthropic()
    messages = [{"role": "user", "content": question}]
    while True:
        resp = client.messages.create(
            model=MODEL, max_tokens=4096, system=SYSTEM,
            tools=TOOLS, messages=messages,
        )
        if resp.stop_reason != "tool_use":
            return next((b.text for b in resp.content if b.type == "text"), "")
        messages.append({"role": "assistant", "content": resp.content})
        messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": b.id,
             "content": rag_search(**b.input)}
            for b in resp.content if b.type == "tool_use"]})


print(research_raw(QUESTION))


## 2 · LangGraph — the loop becomes a declaration

`create_react_agent` collapses the whole `while` loop into one call; `InMemorySaver` makes
Ch 14 checkpointing a **constructor argument**. You gain durable, resumable state per
`thread_id` for free — and you adopt LangGraph's state machine as your execution model. The
tool is just a plain function with a docstring; LangGraph reads its signature into a schema.


In [ ]:
def search_docs(query: str) -> str:
    """Search the company document base; returns passages + source ids."""
    return rag_search(query)  # the Ch 13 retriever


def research_langgraph(question: str, thread_id: str) -> str:
    if MOCK:
        # create_react_agent runs the identical loop internally; thread_id keys the checkpoint.
        _passages = search_docs(CANNED_QUERY)
        return CANNED_ANSWER

    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.prebuilt import create_react_agent

    agent = create_react_agent(
        model=f"anthropic:{MODEL}",
        tools=[search_docs],
        prompt=SYSTEM,
        checkpointer=InMemorySaver(),  # Ch 14 checkpoints, built in
    )
    out = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    return out["messages"][-1].content


print(research_langgraph(QUESTION, thread_id="demo-1"))


## 3 · Pydantic AI — the boundary becomes types

Barely longer than raw, but the contract moves into a type: `CitedAnswer` is **validated** at
the model boundary (`answer`, `source_ids`, `confidence`), auto-retried on a schema failure.
This composes perfectly with the FastAPI layer the capstone sits behind (Part VII).


In [ ]:
from pydantic import BaseModel


class CitedAnswer(BaseModel):
    answer: str
    source_ids: list[str]
    confidence: float  # 0..1, the agent's own calibration


def research_pydantic_ai(question: str) -> CitedAnswer:
    if MOCK:
        # The model's free-text answer arrives parsed + validated into the type.
        _passages = rag_search(CANNED_QUERY)
        return CitedAnswer(
            answer=CANNED_ANSWER,
            source_ids=CANNED_SOURCE_IDS,
            confidence=CANNED_CONFIDENCE,
        )

    from pydantic_ai import Agent

    agent = Agent(
        f"anthropic:{MODEL}",
        output_type=CitedAnswer,
        instructions=SYSTEM,
    )

    @agent.tool_plain
    def search_docs(query: str) -> str:  # noqa: F811  (the typed-boundary version)
        """Search the company document base; returns passages + source ids."""
        return rag_search(query)

    return agent.run_sync(question).output  # validated, or auto-retried


result = research_pydantic_ai(QUESTION)
print(type(result).__name__, "->", result)


## 🔮 Predict before you run

Before running the comparison cell, write down your guesses:

1. Which implementation is the **longest** in lines of code?
2. Which one **hides the most** from a debugger (most behaviour you didn't write)?
3. Which two produce a **plain string** vs. a **validated object**?

Then run the next cell and check yourself.


In [ ]:
import inspect

rows = {
    "raw SDK":      research_raw,
    "LangGraph":    research_langgraph,
    "Pydantic AI":  research_pydantic_ai,
}
hides = {
    "raw SDK":     "nothing — loop, dispatch, history all visible",
    "LangGraph":   "the loop + state machine + checkpoint writes",
    "Pydantic AI": "output parsing, validation, schema-retry",
}
returns = {
    "raw SDK": "str", "LangGraph": "str", "Pydantic AI": "CitedAnswer",
}

print(f"{'framework':<13}{'LoC':>5}  {'returns':<12}  hides from debugger")
print("-" * 78)
for name, fn in rows.items():
    loc = line_count(inspect.getsource(fn))
    print(f"{name:<13}{loc:>5}  {returns[name]:<12}  {hides[name]}")


**What you just saw.** The raw version is the longest and the most honest — every turn is
yours. LangGraph is the shortest *surface* but hides the most machinery behind
`create_react_agent`. Pydantic AI is barely longer than raw yet upgrades the return from a
bare `str` to a machine-checked `CitedAnswer`. Same behaviour; three philosophies. (LoC counts
the full mock+live function body, so the absolute numbers are illustrative — the *ranking* is
the point.)


### One contract, one answer

The diff is meaningful only because the output is identical. Confirm all three converge on the
same cited answer (and the same source id) in mock mode.


In [ ]:
a = research_raw(QUESTION)
b = research_langgraph(QUESTION, thread_id="demo-2")
c = research_pydantic_ai(QUESTION)

assert a == b == c.answer, "mock outputs diverged — the contract is not actually shared"
assert c.source_ids == CANNED_SOURCE_IDS
print("All three agree:\n ", a)


## ⚠️ Pitfall — gentle on the demo, brutal at the edges

The mock above is the *demo*: clean, two-step, identical. Production is the *edge*. The moment
your token bill doubles overnight, the question becomes “what exactly did the framework send to
the model?” — and with the framework versions you can't just read it off your own code.

The cell below stands in for the 2 a.m. scenario: you need the **assembled prompt** (system +
every tool result the loop stitched in). With the raw loop it's in your hands; with a framework
you're reverse-engineering its prompt assembly through callbacks, tracing, or source-reading.


In [ ]:
def assembled_prompt_raw(question: str) -> str:
    """With the raw loop, the wire is right here — you built every line of it."""
    passages = rag_search(CANNED_QUERY)
    return (
        f"SYSTEM: {SYSTEM}\n"
        f"USER: {question}\n"
        f"TOOL search_docs -> {passages}\n"
    )


print(assembled_prompt_raw(QUESTION))
print(
    "With LangGraph / Pydantic AI the same text exists — but you reach it through the "
    "framework's tracing or by reading its source, not your own loop.\n"
    "That gap, multiplied by every weird edge, is the abstraction cost."
)


## 🎯 Senior lens — the capstone's division of labor

A senior doesn't pick one winner; they assign each framework the job it's best at and keep the
reference alive:

- **Pydantic AI for typed single-agent components** — where a validated boundary buys the most
  (anything that feeds the FastAPI layer).
- **LangGraph where durable multi-step orchestration earns its ceremony** — the supervisor
  team, long jobs that pause for approval (Ch 20 interrupts).
- **Raw SDK kept alive in tests and as the reference implementation** — so the team never
  loses the ability to see the wire.

The portable artifacts — your tool/handoff **schemas** (Ch 15/17) and **MCP** (Ch 19) — travel
across all three and across vendor SDKs. Invest in those; treat the orchestration layer as
replaceable. That hexagonal habit (Ch 28) is what turns “rewrite” into “re-wire.”


## Recap

- The **same** `search_docs` research agent runs three ways; in mock mode all three return the
  identical cited answer, so the diff is purely structural.
- **Raw SDK**: longest, most honest — you own the loop, nothing hides.
- **LangGraph**: the loop collapses to `create_react_agent`; checkpointing is a constructor arg.
- **Pydantic AI**: barely longer than raw, but the answer arrives as a validated `CitedAnswer`.
- The cost of magic is the assembled prompt you can no longer read off your own code — weigh
  frameworks by their worst day, not their demo.


## Exercises

Each changes something and asks you to **predict** the result first.

1. **New question, new citation.** Change `QUESTION` to ask about the support SLA. Update the
   canned answer/source id so it cites `doc-sla-02`. Predict: does the *structure* of any of the
   three functions need to change, or only the data? (It should be data-only — that's the point.)
2. **Tighten the type.** Add a `coverage: bool` field to `CitedAnswer` (true when the docs
   covered the question). Predict which of the three implementations is forced to change — and
   why only the typed one gets the new guarantee for free.
3. **Make the retriever miss.** Make `rag_search` return an empty string for an off-topic query
   and have each implementation honour the “say so if uncovered” clause. Predict where the raw
   loop needs an explicit branch and where the framework handles it.
4. **(MOCK=0, optional)** Export `ANTHROPIC_API_KEY`, set `COMPANION_MOCK=0`, and run all three
   live once. Predict: will the three answers be *byte-identical* now? Why not?


In [ ]:
# Exercise 1 — swap the question + citation (data-only change)


In [ ]:
# Exercise 2 — add `coverage: bool` to CitedAnswer; see which impls must change


In [ ]:
# Exercise 3 — empty-result path: honour the 'say so if uncovered' clause


In [ ]:
# Exercise 4 (optional, costs tokens) — COMPANION_MOCK=0 live run


## Next

- **Next notebook:** [`18-02-choosing-a-framework.ipynb`](./18-02-choosing-a-framework.ipynb) —
  turn this hands-on feel into an explicit decision: name the forces, apply the transparency
  test, write the ADR.
- **Blueprints (the production versions of what you just built):**
  [`../../blueprints/agent-loop/`](../../blueprints/agent-loop/) (raw realization) and
  [`../../blueprints/multi-agent-supervisor/`](../../blueprints/multi-agent-supervisor/)
  (framework realization of the same contract).
- **Capstone:** this slice becomes `capstone-project/agents/raw/` (reference), `capstone-project/agents/graph/`
  (LangGraph), and `capstone-project/agents/pydantic_ai/`; checkpoint `checkpoints/ch18-three-ways`.
